### Wortsuche – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021), 1. Runde, Aufgabe 3*

In Rätselheften gibt es oft Rechtecke mit lauter Buchstaben, in welchen man vorgegebene Wörter finden muss. Diese sind
vorwärts oder rückwärts geschrieben in den Zeilen, Spalten und Diagonalen versteckt.

Hier sind zwei Beispiele. Zu finden sind die Wörter VOR, RAD, EVA, TORF.

```
Rätsel 1        Rätsel 2
B V O R C       T F O R A
D T A A G       F D R V R
H V O D I       E V A O E
E J K R L       R O T R T
M N P Q F       D A V A D
```

Viele Menschen würden das zweite Rätsel intuitiv als schwieriger bezeichnen. Ein Grund ist, dass es viele Fragmente der
gesuchten Wörter enthält, während im ersten Rätsel die Wörter sehr hervorstechen.

Es gibt aber auch noch eine Menge weiterer Kriterien, die so ein Rätsel für Menschen leichter oder schwerer erscheinen lassen.

#### Aufgabe 3

Überlege dir verschiedene solche Kriterien und definiere mit deren Hilfe mindestens drei verschiedene Schwierigkeitsgrade.

Schreibe ein Programm, das eine gegebene Wortliste und die Größe des Rechtecks einliest und daraus Buchstabenrechtecke
entsprechend deiner Schwierigkeitsgrade erzeugt.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/worte0.txt` bis `worte5.txt`)

- 1. Zeile: Anzahl der Zeilen h und Spalten b
- 2. Zeile: Anzahl n der Wörter, dann je Zeile ein Wort

`worte0.txt` ist das Beispiel oben (5 × 5; VOR, RAD, EVA, TORF).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zufall und Neustart:** Das Rätsel entsteht durch zufälliges Einfügen. Klappt es nicht, beginnt man einfach von vorn (Las-Vegas-Verfahren).
- **Schwierige Wörter zuerst:** Die längsten Wörter haben die wenigsten Plätze. Wer sie zuerst setzt, läuft seltener in eine Sackgasse.
- **Richtungen als Zahlenpaare:** Eine Richtung ist ein Schritt `(dx, dy)`. Rückwärts ist einfach der entgegengesetzte Schritt.
- **Kontrolle durch Suchen:** Ein Programm, das alle Wörter im fertigen Rätsel *findet*, prüft nach, ob das Rätsel stimmt und ob unbeabsichtigte Treffer drin sind.

**Nützliche Funktionen und Bibliotheken**
- `random.choice(liste)` wählt ein Element zufällig, `random.shuffle(liste)` mischt eine Liste, `random.seed(zahl)` macht den Zufall wiederholbar.
- `text.find(wort, start)` liefert die Position eines Wortes im Text oder `-1`. `wort[::-1]` dreht ein Wort um.
- Eine Liste von Listen (`gitter[zeile][spalte]`) ist ein Rechteck, `set` merkt sich Zellen.

## Die Idee

**Schwierigkeitskriterien.** Was macht ein Rätsel für Menschen schwerer? Wir wählen diese Kriterien:

- **Richtung:** Zeilen von links nach rechts und Spalten von oben nach unten liest man leicht. Diagonalen sind schwerer.
- **Leserichtung:** Rückwärts geschriebene Wörter sind schwerer zu erkennen als vorwärts geschriebene.
- **Auffüllen:** Zufällige Buchstaben fallen zwischen den Wörtern auf. Besteht der Rest aus Buchstaben der gesuchten Wörter, gibt es viele Fast-Treffer (Fragmente).
- **Größe:** Sie ist in der Aufgabe vorgegeben und ändert sich nicht, je größer das Rechteck, desto schwerer.

Daraus werden drei Stufen:

| Stufe | Richtungen | Leserichtung | Rest des Rechtecks |
|---|---|---|---|
| 1 einfach | waagerecht und senkrecht | nur vorwärts (links nach rechts, oben nach unten) | zufällige Buchstaben |
| 2 mittel | waagerecht, senkrecht, beide Diagonalen | vorwärts und rückwärts | zufällige Buchstaben |
| 3 schwer | waagerecht, senkrecht, beide Diagonalen | vorwärts und rückwärts | nur Buchstaben aus den gesuchten Wörtern |

**Das Verfahren.**

1. Die Wörter werden nach Länge sortiert, das längste zuerst. Ist ein Wort länger als `max(h, b)`, gibt es keine Lösung.
2. Für jedes Wort wird eine zufällige **gültige Position** gesucht: Startzelle und Richtung so, dass das Wort im Rechteck bleibt und jede Zelle entweder
   leer ist oder schon den passenden Buchstaben enthält (so dürfen sich Wörter kreuzen).
3. Findet sich für ein Wort keine Position, beginnt alles von vorn, höchstens 10-mal.
4. Alle leeren Zellen werden aufgefüllt: Bei Stufe 1 und 2 mit beliebigen Buchstaben, bei Stufe 3 mit Buchstaben der Wörter.
5. **Wichtig:** Beim Auffüllen können zufällig weitere Vorkommen der gesuchten Wörter entstehen. Das ist bei Stufe 3 sehr wahrscheinlich (bei `worte5.txt`
   besteht das ganze Rechteck aus D, A und S, und das gesuchte Wort ist DAS). Dann wäre die Stufe „schwer“ plötzlich leichter als „einfach“. Deshalb suchen wir nach
   dem Auffüllen alle Wörter im Rechteck und würfeln die Füllbuchstaben jedes **unbeabsichtigten Treffers** neu, bis keiner mehr übrig ist.

## Schritt 1: Einlesen

Die Datei hat in der ersten Zeile `h b`, dann die Anzahl der Wörter und je ein Wort pro Zeile. Wir schreiben alle Wörter groß.

In [ ]:
import random


def lies_woerter(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = [zeile.strip() for zeile in f.read().splitlines() if zeile.strip()]
    h, b = zeilen[0].split()
    n = int(zeilen[1])
    woerter = [w.upper() for w in zeilen[2:2 + n]]
    return int(h), int(b), woerter


h, b, woerter = lies_woerter('beispieldaten/worte1.txt')
print(h, 'Zeilen,', b, 'Spalten, Wörter:', woerter)

6 Zeilen, 6 Spalten, Wörter: ['EIN', 'DA', 'ER', 'INFO', 'DU', 'UND']


## Schritt 2: Richtungen und das Einfügen eines Wortes

Ein Rechteck ist eine Liste von Zeilen, jede Zeile eine Liste von Buchstaben. Leere Zellen enthalten `'.'`.
Eine Richtung ist ein Paar `(dx, dy)`: `dx` ist die Änderung der Spalte, `dy` die Änderung der Zeile bei jedem Buchstaben.

- `(1, 0)` nach rechts, `(0, 1)` nach unten, `(1, 1)` diagonal nach rechts unten, `(1, -1)` diagonal nach rechts oben.
- Alle anderen Richtungen sind die **Umkehrungen** dieser vier, also `(-1, 0)`, `(0, -1)`, `(-1, -1)`, `(-1, 1)`.

`richtungen(stufe)` liefert die erlaubten Richtungen der Stufe. `passt(...)` prüft für eine Startzelle und eine Richtung, ob das Wort hineinpasst, und
`setze(...)` schreibt es hinein.

In [ ]:
def richtungen(stufe):
    if stufe == 1:
        return [(1, 0), (0, 1)]
    vor = [(1, 0), (0, 1), (1, 1), (1, -1)]
    zurueck = [(-dx, -dy) for dx, dy in vor]
    return vor + zurueck


def passt(gitter, wort, i, j, dx, dy):
    """Passt das Wort ab Zeile i, Spalte j in Richtung (dx, dy)? Leer oder gleicher Buchstabe ist erlaubt."""
    h = len(gitter)
    b = len(gitter[0])
    ende_i = i + dy * (len(wort) - 1)
    ende_j = j + dx * (len(wort) - 1)
    if not (0 <= ende_i < h and 0 <= ende_j < b):
        return False                              # ragt aus dem Rechteck
    for k in range(len(wort)):
        feld = gitter[i + dy * k][j + dx * k]
        if feld != '.' and feld != wort[k]:
            return False                          # Kollision mit einem anderen Buchstaben
    return True


def setze(gitter, wort, i, j, dx, dy, fest):
    """Schreibt das Wort ins Rechteck. fest merkt sich die Zellen, die zu einem Wort gehören."""
    for k in range(len(wort)):
        gitter[i + dy * k][j + dx * k] = wort[k]
        fest.add((i + dy * k, j + dx * k))


gitter = [['.'] * 6 for zeile in range(6)]
fest = set()
print(passt(gitter, 'INFO', 0, 0, 1, 0), passt(gitter, 'INFO', 0, 4, 1, 0), passt(gitter, 'INFO', 3, 5, -1, -1))
setze(gitter, 'INFO', 0, 0, 1, 0, fest)
print(passt(gitter, 'UND', 0, 2, 0, 1), passt(gitter, 'FEIN', 0, 2, 0, 1))
for zeile in gitter:
    print(' '.join(zeile))

True False True
False True
I N F O . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .
. . . . . .


## Schritt 3: Alle Wörter einfügen

`fuege_ein(h, b, woerter, stufe)` setzt die Wörter, das längste zuerst. Für jedes Wort sucht die Funktion zuerst bis zu 300-mal eine **zufällige** Startzelle und Richtung.
Ist das erfolglos, zählt sie *alle* gültigen Möglichkeiten auf und nimmt eine davon; gibt es keine, ist dieser Versuch gescheitert (`None`).
Das erste Probieren spart Zeit, denn in einem großen Rechteck ist fast jede zufällige Wahl gültig.
Das Aufzählen aller Möglichkeiten garantiert, dass wir eine Position finden, wenn eine existiert.

In [ ]:
def fuege_ein(h, b, woerter, stufe):
    """Ein Versuch. Liefert (gitter, fest, platzierung) oder None. platzierung: Wort -> (i, j, dx, dy)."""
    gitter = [['.'] * b for zeile in range(h)]
    fest = set()
    platzierung = {}
    erlaubt = richtungen(stufe)
    for wort in sorted(woerter, key=len, reverse=True):
        gefunden = None
        for versuch in range(300):                       # erst zufällig raten
            i = random.randrange(h)
            j = random.randrange(b)
            dx, dy = random.choice(erlaubt)
            if passt(gitter, wort, i, j, dx, dy):
                gefunden = (i, j, dx, dy)
                break
        if gefunden is None:                             # dann alle Möglichkeiten aufzählen
            alle = []
            for i in range(h):
                for j in range(b):
                    for dx, dy in erlaubt:
                        if passt(gitter, wort, i, j, dx, dy):
                            alle.append((i, j, dx, dy))
            if len(alle) == 0:
                return None
            gefunden = random.choice(alle)
        i, j, dx, dy = gefunden
        setze(gitter, wort, i, j, dx, dy, fest)
        platzierung[wort] = gefunden
    return gitter, fest, platzierung


random.seed(1)
gitter, fest, platzierung = fuege_ein(5, 5, ['VOR', 'RAD', 'EVA', 'TORF'], 2)
for zeile in gitter:
    print(' '.join(zeile))
print(platzierung)

. . . A .
. R . V T
. R O E O
. A . V R
. D . . F
{'TORF': (1, 4, 0, 1), 'VOR': (3, 3, -1, -1), 'RAD': (2, 1, 0, 1), 'EVA': (2, 3, 0, -1)}


## Schritt 4: Wörter im Rechteck finden

Die Kontrolle, die auch für das Auffüllen gebraucht wird: `finde_alle` liefert zu jedem Wort alle Vorkommen, jeweils als Menge der Zellen.

Die Idee: In jeder der vier Richtungen zerlegen wir das Rechteck in **Linien** (alle Zeilen, alle Spalten, alle Diagonalen), schreiben die Buchstaben jeder Linie als
Text hin und suchen darin mit `find` das Wort **und das umgedrehte Wort**. Wörter rückwärts zu suchen heißt also, das Wort umzudrehen.
Ein Palindrom wie `LEVEL` liefert dasselbe Vorkommen doppelt, deshalb speichern wir die Zellmengen in einer Menge (`set` von `frozenset`).

In [ ]:
def linien(h, b):
    """Alle Linien in den vier Richtungen als Liste von Zellen (Zeile, Spalte), jeweils von vorn nach hinten."""
    ergebnis = []
    for dx, dy in [(1, 0), (0, 1), (1, 1), (1, -1)]:
        for i in range(h):
            for j in range(b):
                if not (0 <= i - dy < h and 0 <= j - dx < b):     # Startzelle: davor liegt nichts mehr
                    zellen = []
                    ii = i
                    jj = j
                    while 0 <= ii < h and 0 <= jj < b:
                        zellen.append((ii, jj))
                        ii = ii + dy
                        jj = jj + dx
                    ergebnis.append(zellen)
    return ergebnis


def finde_alle(gitter, woerter):
    """Liefert Wort -> Liste aller Vorkommen (jeweils frozenset der Zellen), in allen 8 Richtungen."""
    h = len(gitter)
    b = len(gitter[0])
    treffer = {w: set() for w in woerter}
    for zellen in linien(h, b):
        text = ''.join(gitter[i][j] for i, j in zellen)
        for wort in woerter:
            for suche in [wort, wort[::-1]]:
                start = text.find(suche)
                while start != -1:
                    treffer[wort].add(frozenset(zellen[start:start + len(wort)]))
                    start = text.find(suche, start + 1)
    return {w: list(t) for w, t in treffer.items()}


funde = finde_alle(gitter, ['VOR', 'RAD', 'EVA', 'TORF'])
for wort in funde:
    print(wort, 'kommt', len(funde[wort]), 'mal vor')

VOR kommt 1 mal vor
RAD kommt 1 mal vor
EVA kommt 1 mal vor
TORF kommt 1 mal vor


## Schritt 5: Auffüllen ohne unbeabsichtigte Treffer

Beim Auffüllen zählt die Stufe:

- Stufe 1 und 2: `buchstaben` sind die Buchstaben `A` bis `Z` (zusammen mit allen Buchstaben, die in den Wörtern stehen, damit z. B. Umlaute nicht verraten, wo die Wörter sind).
- Stufe 3: `buchstaben` ist die Liste *aller Buchstaben aller Wörter*. Häufige Buchstaben kommen darin mehrfach vor, also werden sie auch öfter gezogen.

Zuerst werden alle leeren Zellen zufällig gefüllt. Dann wiederholen wir: Wir suchen alle Vorkommen der Wörter (`finde_alle`). Hat ein Vorkommen eine Zelle, die **nicht** zu einem
eingefügten Wort gehört, dann ist es ein unbeabsichtigter Treffer. Wir würfeln die Füllzellen dieses Treffers neu. Das wiederholen wir, bis es keine unbeabsichtigten Treffer mehr gibt (höchstens 1000 Runden).
Vorkommen, die nur aus Zellen eingefügter Wörter bestehen, sind in Ordnung, das sind zum Beispiel die Wörter `ARCHIV` in `ARCHIVBOT`.

In [ ]:
def fuelle_auf(gitter, fest, woerter, stufe):
    """Füllt alle Zellen mit '.' auf. Liefert die Zahl der Runden, oder -1, wenn unbeabsichtigte Treffer bleiben."""
    h = len(gitter)
    b = len(gitter[0])
    alle_buchstaben = ''.join(woerter)
    if stufe == 3:
        buchstaben = alle_buchstaben
    else:
        buchstaben = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ' + alle_buchstaben
    for i in range(h):
        for j in range(b):
            if gitter[i][j] == '.':
                gitter[i][j] = random.choice(buchstaben)
    for runde in range(1000):
        schlecht = set()
        funde = finde_alle(gitter, woerter)
        for wort in woerter:
            for zellen in funde[wort]:
                if not zellen <= fest:                   # mindestens eine Füllzelle dabei
                    schlecht = schlecht | (zellen - fest)
        if len(schlecht) == 0:
            return runde
        for i, j in schlecht:
            gitter[i][j] = random.choice(buchstaben)
    return -1


random.seed(2)
gitter, fest, platzierung = fuege_ein(5, 5, ['VOR', 'RAD', 'EVA', 'TORF'], 3)
print('Runden beim Auffüllen:', fuelle_auf(gitter, fest, ['VOR', 'RAD', 'EVA', 'TORF'], 3))
for zeile in gitter:
    print(' '.join(zeile))

Runden beim Auffüllen: 1
T A O A R
O V F R E
R R T O A
F F D V V
F D A R E


## Schritt 6: Das Rätsel erzeugen

`wortsuche(h, b, woerter, stufe)` fasst alles zusammen:

1. Ist ein Wort länger als `max(h, b)`, gibt es keine Lösung (`None`).
2. Bis zu 10 Versuche: einfügen, dann auffüllen. Gelingt beides, ist das Rätsel fertig.

Zurück kommt das Rechteck als Liste von Zeilen. `zeige` druckt es. `pruefe_raetsel` ist die unabhängige Kontrolle: Jedes Wort kommt vor, und die Zahl seiner Vorkommen
passt zu den Zellen eingefügter Wörter.

In [ ]:
def wortsuche(h, b, woerter, stufe):
    if max(len(w) for w in woerter) > max(h, b):
        return None                                       # kein Wort dieser Länge passt hinein
    for versuch in range(10):
        ergebnis = fuege_ein(h, b, woerter, stufe)
        if ergebnis is None:
            continue
        gitter, fest, platzierung = ergebnis
        if fuelle_auf(gitter, fest, woerter, stufe) >= 0:
            return gitter
    return None


def zeige(gitter):
    for zeile in gitter:
        print(' '.join(zeile))


def pruefe_raetsel(gitter, woerter, stufe):
    """Jedes Wort kommt mindestens einmal vor?"""
    funde = finde_alle(gitter, woerter)
    for wort in woerter:
        if len(funde[wort]) == 0:
            return False
    return True


random.seed(3)
h, b, woerter = lies_woerter('beispieldaten/worte0.txt')
for stufe in [1, 2, 3]:
    print('Stufe', stufe)
    gitter = wortsuche(h, b, woerter, stufe)
    zeige(gitter)
    print('Suche:', ', '.join(woerter), '   geprüft:', pruefe_raetsel(gitter, woerter, stufe))
    print()

Stufe 1
R A D C I
D N E V A
Q O T V E
Y O V O R
T O R F W
Suche: VOR, RAD, EVA, TORF    geprüft: True

Stufe 2
A A R F W
E F V A J
E R R B D
V O S O O
A V V H T
Suche: VOR, RAD, EVA, TORF    geprüft: True

Stufe 3
V O R E F
E D A R R
T V R A O
T D T A T
A F A V E
Suche: VOR, RAD, EVA, TORF    geprüft: True



Wir sehen die Stufen in diesem Beispiel. Bei Stufe 3 besteht das ganze Rechteck nur aus Buchstaben von VOR, RAD, EVA und TORF, und die gesuchten Wörter sind kaum zu sehen.
Die Zufallszahlen sind mit `random.seed` festgelegt, daher kommt immer dieselbe Ausgabe. Bei einem anderen Startwert sieht das Rätsel anders aus.

## Schritt 7: Wie unterscheiden sich die Stufen wirklich?

Wir messen die Schwierigkeit: Dazu zählen wir **Fragmente**, nämlich Stellen im Rechteck (in allen 8 Richtungen), an denen die **ersten zwei Buchstaben** eines gesuchten Wortes stehen,
ohne dass das ganze Wort dort steht. Je mehr Fragmente, desto mehr falsche Fährten. Dazu die **Richtungen** der eingefügten Wörter, hier als Anzahl der Wörter, die
diagonal oder rückwärts stehen. Wir mitteln über je 20 Rätsel zu `worte2.txt` (10 × 10, 6 Wörter).

In [ ]:
def fragmente(gitter, woerter):
    """Zahl der Stellen, an denen die ersten zwei Buchstaben eines Wortes stehen, das Wort aber nicht ganz."""
    h = len(gitter)
    b = len(gitter[0])
    anzahl = 0
    for zellen in linien(h, b):
        text = ''.join(gitter[i][j] for i, j in zellen)
        for wort in woerter:
            for suche in [wort, wort[::-1]]:
                kopf = suche[:2]
                start = text.find(kopf)
                while start != -1:
                    if text[start:start + len(suche)] != suche:
                        anzahl = anzahl + 1
                    start = text.find(kopf, start + 1)
    return anzahl


def schwierige_woerter(h, b, woerter, stufe):
    """Wie viele Wörter stehen diagonal oder rückwärts? (aus einem eigenen Einfügeversuch)"""
    ergebnis = None
    while ergebnis is None:
        ergebnis = fuege_ein(h, b, woerter, stufe)
    anzahl = 0
    for wort, (i, j, dx, dy) in ergebnis[2].items():
        if dy != 0 and dx != 0 or dx < 0 or dy < 0:
            anzahl = anzahl + 1
    return anzahl


random.seed(4)
h, b, woerter = lies_woerter('beispieldaten/worte2.txt')
for stufe in [1, 2, 3]:
    summe_fragmente = 0
    summe_schwer = 0
    for i in range(20):
        gitter = wortsuche(h, b, woerter, stufe)
        summe_fragmente = summe_fragmente + fragmente(gitter, woerter)
        summe_schwer = summe_schwer + schwierige_woerter(h, b, woerter, stufe)
    print('Stufe', stufe, ': Fragmente im Mittel', round(summe_fragmente / 20, 1),
          ', diagonale oder rückwärts stehende Wörter im Mittel', round(summe_schwer / 20, 1), 'von', len(woerter))

Stufe 1 : Fragmente im Mittel 19.6 , diagonale oder rückwärts stehende Wörter im Mittel 0.0 von 6
Stufe 2 : Fragmente im Mittel 19.6 , diagonale oder rückwärts stehende Wörter im Mittel 3.4 von 6
Stufe 3 : Fragmente im Mittel 25.9 , diagonale oder rückwärts stehende Wörter im Mittel 3.6 von 6


Stufe 1 hat keine diagonalen oder rückwärts stehenden Wörter, bei Stufe 2 und 3 sind es im Mittel gut die Hälfte (3,4 und 3,6 von 6 Wörtern). Die Zahl der Fragmente ist bei Stufe 1 und 2 etwa gleich (19,6) und steigt bei Stufe 3 deutlich (25,9).
Das passt zu unseren Kriterien. Stufe 2 und 3 unterscheiden sich im Einfügen nicht, sondern nur im Auffüllen.

## Schritt 8: Alle Beispiele

Für jede Beispieldatei und jede Stufe erzeugen wir ein Rätsel und prüfen, ob alle Wörter drin sind. Gezeigt werden die kleinen Rätsel ganz. Bei den größeren (`worte3.txt` bis `worte5.txt`) zeigen wir
nur einen Ausschnitt, damit die Ausgabe kurz bleibt. Die Datei `worte4.txt` hat 77 Wörter mit 624 Buchstaben in einem 40 × 32-Rechteck (1280 Zellen), das ist schon recht voll.

In [ ]:
for nr in range(6):
    random.seed(100 + nr)
    h, b, woerter = lies_woerter('beispieldaten/worte' + str(nr) + '.txt')
    print('worte' + str(nr) + '.txt:', h, 'x', b, ',', len(woerter), 'Wörter, längstes Wort', max(len(w) for w in woerter))
    for stufe in [1, 2, 3]:
        gitter = wortsuche(h, b, woerter, stufe)
        print('  Stufe', stufe, ': alle Wörter enthalten:', pruefe_raetsel(gitter, woerter, stufe))

worte0.txt: 5 x 5 , 4 Wörter, längstes Wort 4
  Stufe 1 : alle Wörter enthalten: True
  Stufe 2 : alle Wörter enthalten: True
  Stufe 3 : alle Wörter enthalten: True
worte1.txt: 6 x 6 , 6 Wörter, längstes Wort 4
  Stufe 1 : alle Wörter enthalten: True
  Stufe 2 : alle Wörter enthalten: True
  Stufe 3 : alle Wörter enthalten: True
worte2.txt: 10 x 10 , 6 Wörter, längstes Wort 10
  Stufe 1 : alle Wörter enthalten: True
  Stufe 2 : alle Wörter enthalten: True
  Stufe 3 : alle Wörter enthalten: True
worte3.txt: 22 x 24 , 12 Wörter, längstes Wort 12
  Stufe 1 : alle Wörter enthalten: True
  Stufe 2 : alle Wörter enthalten: True
  Stufe 3 : alle Wörter enthalten: True
worte4.txt: 40 x 32 , 77 Wörter, längstes Wort 23
  Stufe 1 : alle Wörter enthalten: True
  Stufe 2 : alle Wörter enthalten: True
  Stufe 3 : alle Wörter enthalten: True
worte5.txt: 30 x 30 , 1 Wörter, längstes Wort 3
  Stufe 1 : alle Wörter enthalten: True
  Stufe 2 : alle Wörter enthalten: True
  Stufe 3 : alle Wörter enthalt

In [ ]:
random.seed(7)
h, b, woerter = lies_woerter('beispieldaten/worte1.txt')
for stufe in [1, 2, 3]:
    print('worte1.txt, Stufe', stufe)
    zeige(wortsuche(h, b, woerter, stufe))
    print('Suche:', ', '.join(woerter))
    print()

worte1.txt, Stufe 1
E D U F D I
I S E J A N
N I H O T F
U N D L G D
O F N M E X
G O F D R O
Suche: EIN, DA, ER, INFO, DU, UND

worte1.txt, Stufe 2
P K I N N E
F Z H I E U
D O E T S D
O N F S R Y
D W B N E D
A D N U I W
Suche: EIN, DA, ER, INFO, DU, UND

worte1.txt, Stufe 3
R D O U E D
O U I N N D
F N D A R D
A O F N I E
E N N I D D
N D D E D D
Suche: EIN, DA, ER, INFO, DU, UND



In [ ]:
random.seed(8)
h, b, woerter = lies_woerter('beispieldaten/worte3.txt')
gitter = wortsuche(h, b, woerter, 3)
print('worte3.txt, Stufe 3 (22 x 24)')
zeige(gitter)
print('Suche:', ', '.join(woerter))

worte3.txt, Stufe 3 (22 x 24)
O K D I O I N A T N K R E E I E O T U R I O E M
O A R I M A G G G K M I I N M I E E O R O H I K
G R E S M O O R I O R F E O L O R N S E I M I S
T K F R A E M P A T H I E R I I O T M T N R M A
F E E R R T L E G I T I M A T I O N K I L M K V
U A R R G E R O M I K M K N S E U N E T K N O K
E C A F O N U R O T U E K S N K M D C H G N M K
O M T R N A O A T E U P I I G R P E M D A K N I
N N E E O I K I V T A M O K O S K I O T I L R L
L R K E M S L V T T E A M C E V K R T R N N N O
V L E T E N T D M U A A T F E I S M N L F O O I
U N N J R I N E N N L S R I V R I T R J E K I E
U U N R T R M K T M N O N U R O M E E N E T T K
E N C K G K E O M K I O V N E F S G O N I U K O
K L H F K N T R L E N N E E O E R U R I G O E N
N A R I O T S A G M M N K M R G E A F C T L F J
I D O R F V A T R N I A O O O E V I K T E O N U
K R N L A O T I N D O O I K O I I C K K T H I N
H U I A M T N O I T I U T N I N N R S G O T E K
L T K G K C K N T T E N C U K J N K N O O O N T
R E T C R 

In [ ]:
random.seed(9)
h, b, woerter = lies_woerter('beispieldaten/worte5.txt')
gitter = wortsuche(h, b, woerter, 3)
print('worte5.txt, Stufe 3 (30 x 30), die ersten 8 Zeilen:')
zeige(gitter[:8])
funde = finde_alle(gitter, woerter)
print('DAS kommt', len(funde['DAS']), 'mal vor')

worte5.txt, Stufe 3 (30 x 30), die ersten 8 Zeilen:
A D D A D D A A A D A A A A S S A A A A A A A A D A A A A A
D S D D S A S D D D A A A A S D S S D D D A A D S D A A D S
D S D D D D A D S D S A A A D S S S S S S A A D D D S D S A
S D A A D D S D S S S S S S S S D S S S A A A D S S S S D A
D A A A A D S D S D D D D D S A A S D S S A A A S D S D S A
S A A A A D D D D D S A S S A A A S S D D A A A S S D D S A
A D A A A A D D D D D D D S A A S S S D S D S A S D S S D D
A D A A A D D D D A A A D D S A S D D D D D S S A A S D D D
DAS kommt 1 mal vor


Bei `worte5.txt` besteht die Stufe 3 aus den drei Buchstaben D, A und S, und das Wort DAS kommt trotzdem nur **einmal** vor, weil wir alle unbeabsichtigten Treffer entfernen.
Ohne diese Kontrolle entstünden bei 900 Zellen mit drei Buchstaben im Mittel gut 200 Treffer (DAS und SAD in vier Richtungen) und die gesuchte Stelle wäre nicht mehr eindeutig.

Und wenn die Wörter nicht passen? Ist ein Wort länger als das Rechteck, meldet `wortsuche` das ehrlich:

In [ ]:
print(wortsuche(3, 3, ['ABCD', 'AB'], 2))
print(wortsuche(3, 3, ['ABC', 'DEF', 'GHI', 'JKL', 'MNO', 'PQR', 'STU', 'VWX', 'YZA', 'BCD'], 1))

None
None


Beide Aufrufe liefern `None`: Im ersten ist ein Wort länger als `max(h, b)`. Im zweiten sind es zwar kurze Wörter, aber zu viele für 9 Zellen.

## Für Neugierige

- **Mehr Kriterien:** Auch die *Wortlänge* (kurze Wörter wie `DA` sind schwer zu finden), Wörter, die sich *kreuzen* (Schnittpunkte), oder Wörter, die in anderen Wörtern *versteckt* sind,
  könnten Stufen unterscheiden. Man könnte eine eigene Kennzahl bauen, die diese Dinge zusammenzählt, und zu einer gewünschten Zahl hin würfeln.
- **Fragmente gezielt einbauen:** Statt nur einzelner Buchstaben könnte Stufe 3 die Anfänge der Wörter (die ersten zwei oder drei Buchstaben) an passenden Stellen im Rest verstreuen, wie im Beispiel *Rätsel 2*.
- **Backtracking:** Wenn ein Wort nicht mehr passt, könnte man das zuletzt gesetzte Wort an eine andere Stelle schieben. Bei den Beispielen reicht aber der einfache Neustart.
- **MiniZinc:** Hier ist ein Constraint-Solver nicht sinnvoll. Das Erzeugen zufälliger Rätsel ist weder eine Such- noch eine Optimierungsaufgabe mit eindeutigem Ergebnis.